In [ ]:
# 1. Install GPU-accelerated Data Science Libraries
!pip install --extra-index-url=https://pypi.nvidia.com cudf-cu12 cuml-cu12

import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import cudf  # GPU Drop-in replacement for Pandas
from cuml.feature_extraction.text import CountVectorizer # GPU Accelerated Tokenizer

# 1. STREAMING THE HUGGING FACE FIN-DATASET
print("Fetching 50,000 real financial rows...")
hf_dataset = load_dataset("mitulshah/global-financial-transaction-classifier", split='train[:50000]')

# Convert to a GPU DataFrame natively. The data moves directly into GPU VRAM.
df_gpu = cudf.DataFrame(hf_dataset)
print("\n--- Raw Ledger Head Sample (Stored in GPU VRAM) ---")
print(df_gpu[['text', 'label']].head())

# 2. VECTORIZING TEXT ENTRIES ON CUDA CORES
# This runs CountVectorizer entirely across thousands of GPU cores instead of 1 CPU thread
gpu_vectorizer = CountVectorizer(max_features=1000, stop_words='english')
X_gpu_sparse = gpu_vectorizer.fit_transform(df_gpu['text'])

# Convert the GPU matrix directly into PyTorch Cupy/DLPack tensors without touching the CPU
X_tensor = torch.as_tensor(X_gpu_sparse.toarray(), device="cuda", dtype=torch.float32)
y_tensor = torch.as_tensor(df_gpu['label'].values, device="cuda", dtype=torch.long)

# 3. HIGH-SPEED CUDA PYTORCH DATALOADER
class GPULedgerDataset(Dataset):
    def __init__(self, features, targets):
        self.X = features
        self.y = targets
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create dataset container already living on the GPU
gpu_dataset = GPULedgerDataset(X_tensor, y_tensor)

# Since data is already sitting in VRAM, num_workers=0 is optimal (no CPU-to-GPU bottleneck)
large_dataloader = DataLoader(gpu_dataset, batch_size=64, shuffle=True)

print(f"\nReady for deep learning! Batches created: {len(large_dataloader)} (64 entries per batch)")